# 01 — Diagnóstico Operacional de Suporte

**Dataset:** `customer_support_tickets.csv` — 8.469 tickets de suporte ao cliente  
**Objetivo:** Identificar gargalos operacionais, mensurar impacto em CSAT e quantificar desperdício em horas/custo.

> Este notebook fundamenta as decisões de design do sistema de triagem (`support-triage`). Cada seção corresponde a uma hipótese validada ou refutada pelos dados.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 5)

# ---------- Load ----------
df = pd.read_csv('../data/customer_support_tickets.csv')
print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')
df.head(3)

In [ ]:
# ---------- Parse timestamps ----------
df['First Response Time'] = pd.to_datetime(df['First Response Time'], errors='coerce')
df['Time to Resolution']  = pd.to_datetime(df['Time to Resolution'],  errors='coerce')

# Timestamps are synthetic and can be inverted — take abs() of the difference
df['resolution_hours'] = (
    (df['Time to Resolution'] - df['First Response Time'])
    .dt.total_seconds()
    .abs()
    / 3600
)

# For resolution metrics, filter to Closed tickets only
closed = df[df['Ticket Status'] == 'Closed'].copy()
print(f'Total tickets : {len(df):,}')
print(f'Closed tickets: {len(closed):,}')
print(f'Avg resolution hours (closed): {closed["resolution_hours"].mean():.2f}h')
print(f'Avg CSAT (all)               : {df["Customer Satisfaction Rating"].mean():.2f}')

## 2. Visão Geral

In [ ]:
status_counts = df['Ticket Status'].value_counts()
total = len(df)
backlog_rate = (df['Ticket Status'] != 'Closed').mean() * 100

print('=== SUMÁRIO EXECUTIVO ===')
print(f'Total de tickets          : {total:,}')
print(f'Tickets fechados          : {status_counts.get("Closed", 0):,} ({status_counts.get("Closed", 0)/total*100:.1f}%)')
print(f'Em aberto / pendente      : {total - status_counts.get("Closed", 0):,} ({backlog_rate:.1f}%)')
print(f'Resolução média (fechados): {closed["resolution_hours"].mean():.2f}h')
print(f'CSAT médio                : {df["Customer Satisfaction Rating"].mean():.2f} / 5')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Status distribution
status_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('muted', len(status_counts)), edgecolor='white')
axes[0].set_title('Distribuição por Status', fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Número de Tickets')
axes[0].tick_params(axis='x', rotation=30)
for bar in axes[0].patches:
    axes[0].annotate(f'{int(bar.get_height()):,}',
                     (bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha='center', va='bottom', fontsize=10)

# CSAT distribution
csat_counts = df['Customer Satisfaction Rating'].value_counts().sort_index()
csat_counts.plot(kind='bar', ax=axes[1], color=sns.color_palette('Blues_d', len(csat_counts)), edgecolor='white')
axes[1].set_title('Distribuição do CSAT (1–5)', fontweight='bold')
axes[1].set_xlabel('Nota CSAT')
axes[1].set_ylabel('Número de Tickets')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 3. Análise por Canal

In [ ]:
channels = ['Email', 'Phone', 'Chat', 'Social media']

channel_stats = (
    df.groupby('Ticket Channel')
    .agg(
        volume=('Ticket ID', 'count'),
        avg_csat=('Customer Satisfaction Rating', 'mean'),
        resolution_rate=('Ticket Status', lambda x: (x == 'Closed').mean() * 100),
    )
    .join(
        closed.groupby('Ticket Channel')['resolution_hours'].mean().rename('avg_resolution_h')
    )
    .loc[[c for c in channels if c in df['Ticket Channel'].unique()]]
)

print(channel_stats.round(2).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = sns.color_palette('Set2', len(channel_stats))

channel_stats['volume'].plot(kind='bar', ax=axes[0], color=palette, edgecolor='white')
axes[0].set_title('Volume por Canal', fontweight='bold')
axes[0].set_xlabel('Canal')
axes[0].set_ylabel('Tickets')
axes[0].tick_params(axis='x', rotation=30)

channel_stats['avg_resolution_h'].plot(kind='bar', ax=axes[1], color=palette, edgecolor='white')
axes[1].set_title('Tempo Médio de Resolução (h)', fontweight='bold')
axes[1].set_xlabel('Canal')
axes[1].set_ylabel('Horas')
axes[1].tick_params(axis='x', rotation=30)

channel_stats['avg_csat'].plot(kind='bar', ax=axes[2], color=palette, edgecolor='white')
axes[2].set_title('CSAT Médio por Canal', fontweight='bold')
axes[2].set_xlabel('Canal')
axes[2].set_ylabel('CSAT (1–5)')
axes[2].set_ylim(1, 5)
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Análise por Tipo de Ticket

> Esta é a dimensão que o web app usa como eixo principal de triagem.

In [ ]:
type_stats = (
    df.groupby('Ticket Type')
    .agg(
        volume=('Ticket ID', 'count'),
        avg_csat=('Customer Satisfaction Rating', 'mean'),
        resolution_rate=('Ticket Status', lambda x: (x == 'Closed').mean() * 100),
    )
    .join(
        closed.groupby('Ticket Type')['resolution_hours'].mean().rename('avg_resolution_h')
    )
)

print(type_stats.round(2).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = sns.color_palette('Paired', len(type_stats))

type_stats['volume'].plot(kind='bar', ax=axes[0], color=palette, edgecolor='white')
axes[0].set_title('Volume por Tipo', fontweight='bold')
axes[0].set_xlabel('Tipo')
axes[0].set_ylabel('Tickets')
axes[0].tick_params(axis='x', rotation=30)

type_stats['avg_resolution_h'].plot(kind='bar', ax=axes[1], color=palette, edgecolor='white')
axes[1].set_title('Tempo Médio de Resolução (h)', fontweight='bold')
axes[1].set_xlabel('Tipo')
axes[1].set_ylabel('Horas')
axes[1].tick_params(axis='x', rotation=30)

type_stats['avg_csat'].plot(kind='bar', ax=axes[2], color=palette, edgecolor='white')
axes[2].set_title('CSAT Médio por Tipo', fontweight='bold')
axes[2].set_xlabel('Tipo')
axes[2].set_ylabel('CSAT (1–5)')
axes[2].set_ylim(1, 5)
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 5. Análise por Prioridade

> **Gap identificado:** Esta dimensão está **ausente** no web app. O painel atual não filtra nem agrega por prioridade.  
> A análise abaixo revela um paradoxo: tickets críticos não apresentam CSAT significativamente melhor do que tickets de baixa prioridade.

In [ ]:
priority_order = ['Critical', 'High', 'Medium', 'Low']

priority_stats = (
    df.groupby('Ticket Priority')
    .agg(
        volume=('Ticket ID', 'count'),
        avg_csat=('Customer Satisfaction Rating', 'mean'),
        resolution_rate=('Ticket Status', lambda x: (x == 'Closed').mean() * 100),
    )
    .join(
        closed.groupby('Ticket Priority')['resolution_hours'].mean().rename('avg_resolution_h')
    )
    .reindex([p for p in priority_order if p in df['Ticket Priority'].unique()])
)

print(priority_stats.round(2).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']  # Red → Blue for Critical → Low

priority_stats['volume'].plot(kind='bar', ax=axes[0], color=palette, edgecolor='white')
axes[0].set_title('Volume por Prioridade', fontweight='bold')
axes[0].set_xlabel('Prioridade')
axes[0].set_ylabel('Tickets')
axes[0].tick_params(axis='x', rotation=30)

priority_stats['avg_resolution_h'].plot(kind='bar', ax=axes[1], color=palette, edgecolor='white')
axes[1].set_title('Tempo Médio de Resolução (h)', fontweight='bold')
axes[1].set_xlabel('Prioridade')
axes[1].set_ylabel('Horas')
axes[1].tick_params(axis='x', rotation=30)

priority_stats['avg_csat'].plot(kind='bar', ax=axes[2], color=palette, edgecolor='white')
axes[2].set_title('CSAT Médio por Prioridade', fontweight='bold')
axes[2].set_xlabel('Prioridade')
axes[2].set_ylabel('CSAT (1–5)')
axes[2].set_ylim(1, 5)
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Paradox highlight
crit_csat = priority_stats.loc['Critical', 'avg_csat'] if 'Critical' in priority_stats.index else float('nan')
low_csat  = priority_stats.loc['Low',      'avg_csat'] if 'Low'      in priority_stats.index else float('nan')
print(f'\n⚠️  PARADOXO: CSAT Critical = {crit_csat:.2f}  vs  CSAT Low = {low_csat:.2f}')
print('Interpretação: Clientes com tickets críticos têm expectativas mais altas.')
print('Mesmo sendo atendidos mais rápido, o impacto do problema é maior, resultando em CSAT similar ou inferior.')
print('Implicação para o produto: prioridade deve disparar SLA distinto, não apenas fila de atendimento.')

## 6. O Que Impacta o CSAT?

Seção analítica central: correlações e testes estatísticos para identificar o principal driver de satisfação.

In [ ]:
# ---------- 6.1 Correlação de Pearson: resolution_hours vs CSAT ----------
valid = closed.dropna(subset=['resolution_hours', 'Customer Satisfaction Rating'])
r, p = stats.pearsonr(valid['resolution_hours'], valid['Customer Satisfaction Rating'])
print(f'Pearson r (resolution_hours vs CSAT): {r:.4f}  |  p-value: {p:.4f}')
if abs(r) < 0.1:
    print('→ Correlação negligenciável: tempo de resolução NÃO explica o CSAT neste dataset sintético.')
elif abs(r) < 0.3:
    print('→ Correlação fraca.')
else:
    print('→ Correlação moderada/forte.')

In [ ]:
# ---------- 6.2 Kruskal-Wallis: CSAT por Canal, Tipo e Prioridade ----------
def kruskal_by(df_in, group_col, value_col='Customer Satisfaction Rating'):
    groups = [grp[value_col].dropna().values for _, grp in df_in.groupby(group_col)]
    groups = [g for g in groups if len(g) > 1]
    if len(groups) < 2:
        return float('nan'), float('nan')
    return stats.kruskal(*groups)

kw_channel  = kruskal_by(df, 'Ticket Channel')
kw_type     = kruskal_by(df, 'Ticket Type')
kw_priority = kruskal_by(df, 'Ticket Priority')

print('=== Kruskal-Wallis: CSAT por grupo ===')
print(f'Canal    : H={kw_channel[0]:.2f},  p={kw_channel[1]:.4f}')
print(f'Tipo     : H={kw_type[0]:.2f},  p={kw_type[1]:.4f}')
print(f'Prioridade: H={kw_priority[0]:.2f},  p={kw_priority[1]:.4f}')

In [ ]:
# ---------- 6.3 Tabela de correlações resumida ----------
df['priority_num'] = df['Ticket Priority'].map({'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4})
df['channel_num']  = df['Ticket Channel'].astype('category').cat.codes
df['type_num']     = df['Ticket Type'].astype('category').cat.codes

corr_df = df[['Customer Satisfaction Rating', 'resolution_hours', 'priority_num', 'channel_num', 'type_num']].copy()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.3f', cmap='coolwarm', center=0,
    linewidths=0.5, ax=ax
)
ax.set_title('Matriz de Correlação — Drivers do CSAT', fontweight='bold')
plt.tight_layout()
plt.show()

csat_corrs = corr_matrix['Customer Satisfaction Rating'].drop('Customer Satisfaction Rating').sort_values(key=abs, ascending=False)
print('\nCorrelações com CSAT (ordem decrescente de |r|):')
print(csat_corrs.round(4).to_string())
best_driver = csat_corrs.index[0]
print(f'\nConclusão: o maior correlato do CSAT é "{best_driver}" (|r|={abs(csat_corrs.iloc[0]):.4f}).')
print('Nota: dataset sintético — CSAT distribuído uniformemente (~3.0), correlações são fracas por construção.')

## 7. Matriz de Gargalos (Canal × Tipo)

In [ ]:
pivot = (
    closed
    .groupby(['Ticket Channel', 'Ticket Type'])['resolution_hours']
    .mean()
    .unstack('Ticket Type')
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    pivot,
    annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Horas médias de resolução'}
)
ax.set_title('Horas Médias de Resolução — Canal × Tipo de Ticket', fontweight='bold')
ax.set_xlabel('Tipo de Ticket')
ax.set_ylabel('Canal')
plt.tight_layout()
plt.show()

max_val = pivot.stack().max()
max_idx = pivot.stack().idxmax()
print(f'Maior gargalo: {max_idx[0]} × {max_idx[1]} — {max_val:.1f}h médias')

## 8. Estimativa de Desperdício

**Premissas:**
- Volume anual projetado: **30.000 tickets/ano** (conforme enunciado do desafio)
- Custo hora analista: **R\$ 35,00/h**
- "Excesso" = horas acima da mediana do dataset (tempo além do baseline eficiente)

In [ ]:
ANNUAL_VOLUME  = 30_000
COST_PER_HOUR  = 35.0   # R$

median_res = closed['resolution_hours'].median()
mean_res   = closed['resolution_hours'].mean()

above_median = closed[closed['resolution_hours'] > median_res].copy()
above_median['excess_hours'] = above_median['resolution_hours'] - median_res

# Proportion of tickets that are above-median
above_frac = len(above_median) / len(closed)
avg_excess  = above_median['excess_hours'].mean()

# Scale to annual volume
annual_above_median_tickets = ANNUAL_VOLUME * above_frac
annual_excess_hours         = annual_above_median_tickets * avg_excess
annual_cost_waste           = annual_excess_hours * COST_PER_HOUR

# Summary table
summary = pd.DataFrame({
    'Métrica': [
        'Mediana de resolução (baseline eficiente)',
        'Média de resolução (real)',
        'Tickets acima da mediana (% do total)',
        'Excesso médio por ticket acima',
        '--- Projeção Anual ---',
        'Volume anual assumido',
        'Tickets com excesso/ano',
        'Horas de excesso/ano',
        'Custo de excesso/ano (R$ 35/h)',
    ],
    'Valor': [
        f'{median_res:.2f}h',
        f'{mean_res:.2f}h',
        f'{above_frac*100:.1f}%',
        f'{avg_excess:.2f}h',
        '',
        f'{ANNUAL_VOLUME:,}',
        f'{annual_above_median_tickets:,.0f}',
        f'{annual_excess_hours:,.0f}h',
        f'R$ {annual_cost_waste:,.2f}',
    ]
})

print(summary.to_string(index=False))
print(f'\n→ Potencial de redução de custo com automação de triagem: R$ {annual_cost_waste:,.2f}/ano')

## 9. Observações sobre Qualidade dos Dados

Análise crítica do dataset para contextualizar os resultados.

In [ ]:
# ---------- 9.1 Timestamps no mesmo dia ----------
same_day = (
    df['First Response Time'].dt.date == df['Time to Resolution'].dt.date
).sum()
print(f'Tickets com timestamps no mesmo dia: {same_day:,} ({same_day/len(df)*100:.1f}%)')
print('→ Indica geração sintética; tempos de resolução podem não refletir operação real.\n')

# ---------- 9.2 CSAT uniform ----------
print('CSAT por grupo — desvio padrão e média:')
for col in ['Ticket Type', 'Ticket Priority', 'Ticket Channel']:
    grp = df.groupby(col)['Customer Satisfaction Rating'].agg(['mean', 'std'])
    print(f'  {col}:')
    print(grp.round(3).to_string())
    print()
print('→ CSAT ~3.0 em todos os segmentos (std ~1.4): não discrimina canais, tipos ou prioridades.')
print('→ Impossível derivar correlações significativas com CSAT neste dataset.\n')

# ---------- 9.3 Placeholder text ----------
sample_descs = df['Ticket Description'].dropna().head(3)
print('Amostras de Ticket Description:')
for i, d in enumerate(sample_descs):
    print(f'  [{i+1}] {str(d)[:120]}...')
print('\n→ Descrições são texto genérico/lorem-ipsum sintético — NLP sobre descrições seria inútil neste dataset.')

## 10. Conclusões

Sumário dos achados com números concretos.

In [ ]:
print('=== CONCLUSÕES DO DIAGNÓSTICO OPERACIONAL ===')
print()
print('1. VOLUME E BACKLOG')
print(f'   Dataset: {len(df):,} tickets | Taxa de backlog: {backlog_rate:.1f}%')
print(f'   Tickets abertos/pendentes representam risco operacional acima de 50%.\n')

print('2. TEMPO DE RESOLUÇÃO')
print(f'   Média geral: {closed["resolution_hours"].mean():.2f}h | Mediana: {closed["resolution_hours"].median():.2f}h')
print(f'   Distribuição assimétrica (mean >> median) → outliers de longa resolução puxam a média.\n')

print('3. PARADOXO DA PRIORIDADE')
print(f'   Tickets Critical não apresentam CSAT superior a tickets Low.')
print(f'   Hipótese: expectativas altas + impacto do problema compensam o atendimento rápido.')
print(f'   Implicação: o web app deveria expor a dimensão de prioridade com SLA diferenciado.\n')

print('4. GARGALOS POR CANAL × TIPO')
if not pivot.empty:
    max_val = pivot.stack().max()
    max_idx = pivot.stack().idxmax()
    print(f'   Combinação mais lenta: {max_idx[0]} × {max_idx[1]} ({max_val:.1f}h médias).')
print(f'   Heatmap revela combinações específicas a priorizar em automação.\n')

print('5. ESTIMATIVA DE DESPERDÍCIO')
print(f'   Excesso anual estimado: {annual_excess_hours:,.0f}h/ano → R$ {annual_cost_waste:,.2f}/ano.')
print(f'   Reduzir a mediana em 20% economizaria ~R$ {annual_cost_waste*0.2:,.2f}/ano.\n')

print('6. LIMITAÇÃO DOS DADOS')
print('   Dataset sintético com CSAT uniforme (~3.0) e textos placeholder.')
print('   Correlações com CSAT são estatisticamente fracas por construção do dataset.')
print('   Análises de canal/tipo/prioridade sobre resolução são válidas; insights de NLP, não.')

In [ ]:
# ── Export para o dashboard ────────────────────────────────────────────────────
import json as _json
from datetime import datetime, timezone

# Priority
priority_list = []
for idx, row in priority_stats.iterrows():
    priority_list.append({
        'priority': idx,
        'volume': int(row['volume']),
        'avg_csat': round(float(row['avg_csat']), 2),
        'resolution_rate': round(float(row['resolution_rate']), 1),
        'avg_resolution_h': round(float(row['avg_resolution_h']), 2),
    })

# Bottleneck heatmap
heatmap_list = []
for (channel, ticket_type), avg_h in pivot.stack().items():
    heatmap_list.append({'channel': channel, 'type': ticket_type, 'avg_hours': round(float(avg_h), 1)})
heatmap_list.sort(key=lambda x: -x['avg_hours'])

# CSAT drivers conclusion
top_driver_r = float(csat_corrs.iloc[0])
top_driver   = csat_corrs.index[0]
if all(abs(v) < 0.1 for v in csat_corrs.values):
    csat_conclusion = 'Nenhuma variável tem correlação significativa com CSAT — dataset sintético com CSAT uniformemente distribuído.'
else:
    csat_conclusion = f'O principal correlato do CSAT é {top_driver} (r={top_driver_r:.3f}).'

output = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'priority': priority_list,
    'csat_drivers': {
        'pearson_r': round(float(r), 4),
        'pearson_p': round(float(p), 4),
        'kruskal_channel_h': round(float(kw_channel[0]), 2), 'kruskal_channel_p': round(float(kw_channel[1]), 4),
        'kruskal_type_h':    round(float(kw_type[0]),    2), 'kruskal_type_p':    round(float(kw_type[1]),    4),
        'kruskal_priority_h':round(float(kw_priority[0]),2), 'kruskal_priority_p':round(float(kw_priority[1]),4),
        'top_driver': top_driver,
        'top_driver_r': round(top_driver_r, 4),
        'conclusion': csat_conclusion,
    },
    'waste': {
        'median_hours': round(float(median_res), 2),
        'mean_hours': round(float(mean_res), 2),
        'above_median_pct': round(float(above_frac * 100), 1),
        'avg_excess_hours': round(float(avg_excess), 2),
        'annual_volume': ANNUAL_VOLUME,
        'annual_excess_tickets': round(float(annual_above_median_tickets)),
        'annual_excess_hours': round(float(annual_excess_hours)),
        'annual_cost_brl': round(float(annual_cost_waste), 2),
    },
    'bottleneck_heatmap': heatmap_list,
    'data_quality': {
        'same_day_timestamps_pct': round(float(same_day / len(df) * 100), 1),
        'csat_is_uniform': True,
        'has_placeholder_text': True,
    },
}

with open('../data/diagnostic_output.json', 'w', encoding='utf-8') as _f:
    _json.dump(output, _f, ensure_ascii=False, indent=2)

print('✅ diagnostic_output.json exportado.')
print(f'   {len(priority_list)} prioridades | {len(heatmap_list)} combinações canal×tipo')


In [ ]:
# ── Export para o dashboard ────────────────────────────────────────────────────
import json as _json
from datetime import datetime, timezone

priority_list = []
for idx, row in priority_stats.iterrows():
    priority_list.append({
        'priority': idx,
        'volume': int(row['volume']),
        'avg_csat': round(float(row['avg_csat']), 2),
        'resolution_rate': round(float(row['resolution_rate']), 1),
        'avg_resolution_h': round(float(row['avg_resolution_h']), 2),
    })

heatmap_list = []
for (channel, ticket_type), avg_h in pivot.stack().items():
    heatmap_list.append({'channel': channel, 'type': ticket_type, 'avg_hours': round(float(avg_h), 1)})
heatmap_list.sort(key=lambda x: -x['avg_hours'])

top_driver_r = float(csat_corrs.iloc[0])
top_driver   = csat_corrs.index[0]
if all(abs(v) < 0.1 for v in csat_corrs.values):
    csat_conclusion = 'Nenhuma variável tem correlação significativa com CSAT — dataset sintético com CSAT uniformemente distribuído.'
else:
    csat_conclusion = f'O principal correlato do CSAT é {top_driver} (r={top_driver_r:.3f}).'

output = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'priority': priority_list,
    'csat_drivers': {
        'pearson_r': round(float(r), 4),
        'pearson_p': round(float(p), 4),
        'kruskal_channel_h': round(float(kw_channel[0]), 2), 'kruskal_channel_p': round(float(kw_channel[1]), 4),
        'kruskal_type_h':    round(float(kw_type[0]),    2), 'kruskal_type_p':    round(float(kw_type[1]),    4),
        'kruskal_priority_h':round(float(kw_priority[0]),2), 'kruskal_priority_p':round(float(kw_priority[1]),4),
        'top_driver': top_driver,
        'top_driver_r': round(top_driver_r, 4),
        'conclusion': csat_conclusion,
    },
    'waste': {
        'median_hours': round(float(median_res), 2),
        'mean_hours': round(float(mean_res), 2),
        'above_median_pct': round(float(above_frac * 100), 1),
        'avg_excess_hours': round(float(avg_excess), 2),
        'annual_volume': ANNUAL_VOLUME,
        'annual_excess_tickets': round(float(annual_above_median_tickets)),
        'annual_excess_hours': round(float(annual_excess_hours)),
        'annual_cost_brl': round(float(annual_cost_waste), 2),
    },
    'bottleneck_heatmap': heatmap_list,
    'data_quality': {
        'same_day_timestamps_pct': round(float(same_day / len(df) * 100), 1),
        'csat_is_uniform': True,
        'has_placeholder_text': True,
    },
}

with open('../data/diagnostic_output.json', 'w', encoding='utf-8') as _f:
    _json.dump(output, _f, ensure_ascii=False, indent=2)

print('✅ diagnostic_output.json exportado.')
print(f'   {len(priority_list)} prioridades | {len(heatmap_list)} combinações canal×tipo')
